## Feature Engineering for the FMP Data ##

### Loading Data and Import Libraries ###

In [4]:
import pandas as pd
import numpy as np
import os 

print(os.getcwd())

# Read in the txt tickers file
txt_path = f'../../data/raw_data/tickers_448_v2.txt'
with open(txt_path, 'r') as f:
    tickers = f.read().splitlines()
    
    
# Reading in the raw data 
full_income_statement = pd.read_parquet('../../data/raw_data/full_income_statement.parquet')
full_balance_sheet = pd.read_parquet('../../data/raw_data/full_balance_sheet.parquet')
full_cash_flow = pd.read_parquet('../../data/raw_data/full_cash_flow.parquet')
full_financial_ratios = pd.read_parquet('../../data/raw_data/full_financial_ratios.parquet')
full_williams = pd.read_parquet('../../data/raw_data/full_williams.parquet')
all_insider_trades = pd.read_parquet('../../data/raw_data/all_insider_trades.parquet')
full_economic_data = pd.read_parquet('../../data/raw_data/full_economic_data.parquet')

c:\Users\yinki\OneDrive\NUS\BT4101\fyp-kiat\ML_Core\src\utils


### Engineering Full Income Statement ###

In [5]:
## Downloading forex rates 
import yfinance as yf

# Feature Engineering for Income Statement
value_cols = ['revenue', 'costOfRevenue',
       'grossProfit', 'researchAndDevelopmentExpenses',
       'generalAndAdministrativeExpenses', 'sellingAndMarketingExpenses',
       'sellingGeneralAndAdministrativeExpenses', 'otherExpenses',
       'operatingExpenses', 'costAndExpenses', 'netInterestIncome',
       'interestIncome', 'interestExpense', 'depreciationAndAmortization',
       'ebitda', 'ebit', 'nonOperatingIncomeExcludingInterest',
       'operatingIncome', 'totalOtherIncomeExpensesNet', 'incomeBeforeTax',
       'incomeTaxExpense', 'netIncomeFromContinuingOperations',
       'netIncomeFromDiscontinuedOperations', 'otherAdjustmentsToNetIncome',
       'netIncome', 'netIncomeDeductions', 'bottomLineNetIncome', 'eps',
       'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil']

# 2. Organize tickers into a dictionary for easier looping
fx_tickers = {
    'SGD': 'SGDUSD=X', 'JPY': 'JPYUSD=X', 'EUR': 'EURUSD=X',
    'CAD': 'CADUSD=X', 'AUD': 'AUDUSD=X', 'CNY': 'CNYUSD=X', 'DKK': 'DKKUSD=X', 
}

fx_ticker_data = {
    'SGD': None, 'JPY': None, 'EUR': None,
    'CAD': None, 'AUD': None, 'CNY': None, 'DKK': None,
}
full_income_statement['fx_rate'] = 1.0

# 3. Efficiently download and map the reciprocal rate
for currency, ticker in fx_tickers.items():
    data = yf.download(ticker, start="2000-01-01", end="2026-01-01")
    fx_ticker_data[currency] = data
    
    # Map the date to the 'Close' price and take the reciprocal (1 / rate)
    # We use .ffill() to handle dates where the market was closed (weekends/holidays)
    full_income_statement.loc[full_income_statement['reportedCurrency'] == currency, 'fx_rate'] = (
        1 / full_income_statement.loc[full_income_statement['reportedCurrency'] == currency, 'date'].map(data['Close'].squeeze())
        ).ffill()
    
    
for col in value_cols:
    full_income_statement[f'{col}_usd'] = full_income_statement[col] * full_income_statement['fx_rate']

full_income_statement = full_income_statement.rename(columns={'date': f'Date', 'symbol': 'Ticker'})
                                                

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


### Engineering Full Balance Sheet ###

In [6]:
balance_sheet_value_cols = [
    'cashAndCashEquivalents',
       'shortTermInvestments', 'cashAndShortTermInvestments', 'netReceivables',
       'accountsReceivables', 'otherReceivables', 'inventory', 'prepaids',
       'otherCurrentAssets', 'totalCurrentAssets', 'propertyPlantEquipmentNet',
       'goodwill', 'intangibleAssets', 'goodwillAndIntangibleAssets',
       'longTermInvestments', 'taxAssets', 'otherNonCurrentAssets',
       'totalNonCurrentAssets', 'otherAssets', 'totalAssets', 'totalPayables',
       'accountPayables', 'otherPayables', 'accruedExpenses', 'shortTermDebt',
       'capitalLeaseObligationsCurrent', 'taxPayables', 'deferredRevenue',
       'otherCurrentLiabilities', 'totalCurrentLiabilities', 'longTermDebt',
       'capitalLeaseObligationsNonCurrent', 'deferredRevenueNonCurrent',
       'deferredTaxLiabilitiesNonCurrent', 'otherNonCurrentLiabilities',
       'totalNonCurrentLiabilities', 'otherLiabilities',
       'capitalLeaseObligations', 'totalLiabilities', 'treasuryStock',
       'preferredStock', 'commonStock', 'retainedEarnings',
       'additionalPaidInCapital', 'accumulatedOtherComprehensiveIncomeLoss',
       'otherTotalStockholdersEquity', 'totalStockholdersEquity',
       'totalEquity', 'minorityInterest', 'totalLiabilitiesAndTotalEquity',
       'totalInvestments', 'totalDebt', 'netDebt'
]

full_balance_sheet['fx_rate'] = 1.0
for currency, ticker in fx_tickers.items():
    data = fx_ticker_data[currency]
    # Map the date to the 'Close' price and take the reciprocal (1 / rate)
    # We use .ffill() to handle dates where the market was closed (weekends/holidays)
    full_balance_sheet.loc[full_balance_sheet['reportedCurrency'] == currency, 'fx_rate'] = (
        1 / full_balance_sheet.loc[full_balance_sheet['reportedCurrency'] == currency, 'date'].map(data['Close'].squeeze())
        ).ffill()
    

for col in balance_sheet_value_cols:
    full_balance_sheet[f'{col}_usd'] = full_balance_sheet[col] * full_balance_sheet['fx_rate']

full_balance_sheet = full_balance_sheet.rename(columns={'date': f'Date', 'symbol': 'Ticker'})
    


### Engineering Full Cash Flow ###

In [7]:
cash_flow_value_cols = [
    'netIncome',
       'depreciationAndAmortization', 'deferredIncomeTax',
       'stockBasedCompensation', 'changeInWorkingCapital',
       'accountsReceivables', 'inventory', 'accountsPayables',
       'otherWorkingCapital', 'otherNonCashItems',
       'netCashProvidedByOperatingActivities',
       'investmentsInPropertyPlantAndEquipment', 'acquisitionsNet',
       'purchasesOfInvestments', 'salesMaturitiesOfInvestments',
       'otherInvestingActivities', 'netCashProvidedByInvestingActivities',
       'netDebtIssuance', 'longTermNetDebtIssuance',
       'shortTermNetDebtIssuance', 'netStockIssuance',
       'netCommonStockIssuance', 'commonStockIssuance',
       'commonStockRepurchased', 'netPreferredStockIssuance',
       'netDividendsPaid', 'commonDividendsPaid', 'preferredDividendsPaid',
       'otherFinancingActivities', 'netCashProvidedByFinancingActivities',
       'effectOfForexChangesOnCash', 'netChangeInCash', 'cashAtEndOfPeriod',
       'cashAtBeginningOfPeriod', 'operatingCashFlow', 'capitalExpenditure',
       'freeCashFlow', 'incomeTaxesPaid', 'interestPaid']



full_cash_flow['fx_rate'] = 1.0
for currency, ticker in fx_tickers.items():
    data = fx_ticker_data[currency]
    # Map the date to the 'Close' price and take the reciprocal (1 / rate)
    # We use .ffill() to handle dates where the market was closed (weekends/holidays)
    full_cash_flow.loc[full_cash_flow['reportedCurrency'] == currency, 'fx_rate'] = (
        1 / full_cash_flow.loc[full_cash_flow['reportedCurrency'] == currency, 'date'].map(data['Close'].squeeze())
        ).ffill()
    
for col in cash_flow_value_cols:
    full_cash_flow[f'{col}_usd'] = full_cash_flow[col] * full_cash_flow['fx_rate']
full_cash_flow = full_cash_flow.rename(columns={'date': f'Date', 'symbol': 'Ticker'})


### Merge all the usd columns together by date and symbol ###

In [8]:
full_financials = full_income_statement.merge(
    full_balance_sheet,
    on=['Ticker', 'Date'],
    suffixes=('_income_statement', '_balance_sheet')
).merge(
    full_cash_flow,
    on=['Ticker', 'Date'],
    suffixes=('', '_cash_flow')
)

full_financials = full_financials[['Date', 'Ticker'] + [col for col in full_financials.columns if col.endswith('_usd')]]
full_financials = full_financials.rename(columns={col: col.replace('_usd', '') for col in full_financials.columns if col.endswith('_usd')})
full_financials['Date'] = pd.to_datetime(full_financials['Date']).dt.strftime("%Y-%m-%d")
full_financials.to_parquet('../../data/processed_data/full_financials_usd.parquet', index=False)

### Engineering Full Financial Ratios ###

In [9]:
full_financial_ratios = full_financial_ratios.rename(columns={'date': 'Date', 'symbol': 'Ticker'})
full_financial_ratios['Date'] = pd.to_datetime(full_financial_ratios['Date']).dt.strftime("%Y-%m-%d")
full_financial_ratios

,Ticker,Date,fiscalYear,period,reportedCurrency,grossProfitMargin,ebitMargin,ebitdaMargin,operatingProfitMargin,pretaxProfitMargin,...,operatingCashFlowPerShare,capexPerShare,freeCashFlowPerShare,netIncomePerEBT,ebtPerEbit,priceToFairValue,debtToMarketCap,effectiveTaxRate,enterpriseValueMultiple,dividendPerShare
0,JNJ,2024-12-29,2024,FY,USD,0.690715,0.196372,0.278999,0.249367,0.187872,...,10.080173,1.837744,8.242429,0.842932,0.753397,4.826722,0.106166,0.157068,14.430063,4.911311
1,JNJ,2023-12-31,2023,FY,USD,0.688195,0.185935,0.273841,0.274886,0.176869,...,8.995856,1.793172,7.202684,2.333887,0.643428,5.773996,0.073865,0.115257,17.348790,4.645747
2,JNJ,2022-12-31,2022,FY,USD,0.692512,0.245468,0.332604,0.262695,0.242018,...,8.109131,1.533901,6.575229,0.926752,0.921287,6.011300,0.085862,0.154398,18.359140,4.469702
3,JNJ,2021-12-31,2021,FY,USD,0.702794,0.245885,0.339738,0.265977,0.243561,...,8.904645,1.389140,7.515505,1.088643,0.915724,6.075639,0.075046,0.071801,17.532094,4.196328
4,JNJ,2020-12-31,2020,FY,USD,0.655781,0.202194,0.289753,0.238945,0.199760,...,8.940510,1.271409,7.669101,0.891920,0.836011,6.547374,0.085121,0.108080,18.203257,3.981368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15370,PCAR,1989-12-31,1989,FY,USD,0.202220,0.103438,0.118397,0.103438,0.103125,...,0.461310,0.075603,0.385707,0.665841,0.996981,1.492377,0.825001,0.334159,5.481591,0.145315
15371,PCAR,1988-12-31,1988,FY,USD,0.162618,0.067284,0.083478,0.067284,0.086016,...,0.000000,0.000000,0.000000,0.656705,1.278415,1.625663,0.386551,0.343295,6.456213,0.111902
15372,PCAR,1987-12-31,1987,FY,USD,0.158980,0.067154,0.079041,0.067154,0.071811,...,0.000000,0.000000,0.000000,0.639932,1.069343,1.275244,0.046582,0.360068,3.799242,0.013169
15373,PCAR,1986-12-31,1986,FY,USD,0.126587,0.033693,0.046335,0.033693,0.043495,...,0.000000,0.000000,0.000000,0.695262,1.290909,1.149252,0.022262,0.304738,10.002477,0.000000


### Williams engineering ##

In [10]:
full_williams = full_williams.rename(columns={'date': 'Date', 'symbol': 'Ticker'})
full_williams['Date'] = pd.to_datetime(full_williams['Date']).dt.strftime("%Y-%m-%d")
full_williams.to_parquet('../../data/processed_data/full_williams.parquet', index=False)

### Engineering Insider Trades ###

In [11]:

all_insider_trades['securitiesTransacted'] = np.where(all_insider_trades['acquisitionOrDisposition'] == 'A',
                                                     all_insider_trades['securitiesTransacted'],
                                                        -all_insider_trades['securitiesTransacted'])
all_insider_trades['totalMarketValue'] = all_insider_trades['securitiesTransacted'] * all_insider_trades['price']

# Aggregate by symbol and filingDate
insider_summary = all_insider_trades.groupby(['symbol', 'filingDate']).agg(
    totalSecuritiesTransacted=('securitiesTransacted', 'sum'),
    totalMarketValue=('totalMarketValue', 'sum'),
    numTransactions=('securitiesTransacted', 'count')
).reset_index()
insider_summary = insider_summary.rename(columns={'symbol': 'Ticker', 'filingDate': 'Date'})
insider_summary['Date'] = pd.to_datetime(insider_summary['Date']).dt.strftime("%Y-%m-%d")
insider_summary.to_parquet('../../data/processed_data/insider_summary.parquet', index=False)

In [12]:
insider_summary.columns

Index(['Ticker', 'Date', 'totalSecuritiesTransacted', 'totalMarketValue',
       'numTransactions'],
      dtype='object')

### Engineering Full Economic Data ###

In [13]:
full_economic_data = full_economic_data.sort_values(by=['date']).ffill()
full_economic_data = full_economic_data.rename(columns={'date': 'Date'})
full_economic_data['Date'] = pd.to_datetime(full_economic_data['Date']).dt.strftime("%Y-%m-%d")
full_economic_data.to_parquet('../../data/processed_data/full_economic_data.parquet', index=False)

In [14]:
full_economic_data

,Date,GDP,realGDP,nominalPotentialGDP,realGDPPerCapita,federalFunds,CPI,inflationRate,inflation,retailSales,...,industrialProductionTotalIndex,newPrivatelyOwnedHousingUnitsStartedTotalUnits,totalVehicleSales,retailMoneyFunds,smoothedUSRecessionProbabilities,3MonthOr90DayRatesAndYieldsCertificatesOfDeposit,commercialBankInterestRateOnCreditCardPlansAllAccounts,30YearFixedRateMortgageAverage,15YearFixedRateMortgageAverage,tradeBalanceGoodsAndServices
0,2000-01-01,10002.179,13878.147,9837.79823,49335.0,5.45,169.30,NaN,3.376857,237541.0,...,91.5380,1636.0,18.635,840.7,0.26,5.95,NaN,NaN,NaN,-27131.0
1,2000-01-07,10002.179,13878.147,9837.79823,49335.0,5.45,169.30,NaN,3.376857,237541.0,...,91.5380,1636.0,18.635,840.7,0.26,5.95,NaN,8.15,7.73,-27131.0
2,2000-01-08,10002.179,13878.147,9837.79823,49335.0,5.45,169.30,NaN,3.376857,237541.0,...,91.5380,1636.0,18.635,840.7,0.26,5.95,NaN,8.15,7.73,-27131.0
3,2000-01-14,10002.179,13878.147,9837.79823,49335.0,5.45,169.30,NaN,3.376857,237541.0,...,91.5380,1636.0,18.635,840.7,0.26,5.95,NaN,8.18,7.78,-27131.0
4,2000-01-15,10002.179,13878.147,9837.79823,49335.0,5.45,169.30,NaN,3.376857,237541.0,...,91.5380,1636.0,18.635,840.7,0.26,5.95,NaN,8.18,7.78,-27131.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7173,2025-05-28,30485.729,23770.976,29836.74000,69499.0,4.33,320.58,2.32,2.949525,618037.0,...,100.9655,1282.0,16.067,2111.4,0.56,5.32,21.16,6.86,6.01,-71052.0
7174,2025-05-29,30485.729,23770.976,29836.74000,69499.0,4.33,320.58,2.32,2.949525,618037.0,...,100.9655,1282.0,16.067,2111.4,0.56,5.32,21.16,6.89,6.03,-71052.0
7175,2025-05-30,30485.729,23770.976,29836.74000,69499.0,4.33,320.58,2.34,2.949525,618037.0,...,100.9655,1282.0,16.067,2111.4,0.56,5.32,21.16,6.89,6.03,-71052.0
7176,2025-05-31,30485.729,23770.976,29836.74000,69499.0,4.33,320.58,2.34,2.949525,618037.0,...,100.9655,1282.0,16.067,2111.4,0.56,5.32,21.16,6.89,6.03,-71052.0


### Historical Analyst Grades ###

In [15]:
historical_analyst_grades = pd.read_parquet('../../data/raw_data/full_historical_analyst_grades.parquet')
historical_analyst_grades = historical_analyst_grades.rename(columns={'date': 'Date', 'symbol': 'Ticker'})
historical_analyst_grades['analyst_score'] = historical_analyst_grades['analystRatingsBuy'] * 1 + \
                                            historical_analyst_grades['analystRatingsHold'] * 0 + \
                                            historical_analyst_grades['analystRatingsSell'] * -1 + \
                                            historical_analyst_grades['analystRatingsStrongBuy'] * 2 + \
                                            historical_analyst_grades['analystRatingsStrongSell'] * -2
historical_analyst_grades['Date'] = pd.to_datetime(historical_analyst_grades['Date']).dt.strftime("%Y-%m-%d")
historical_analyst_grades.to_parquet('../../data/processed_data/historical_analyst_grades.parquet', index=False)
                                            


In [16]:
historical_analyst_grades

,Ticker,Date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,analyst_score
0,JNJ,2026-01-01,4,9,11,0,1,15
1,JNJ,2025-12-01,4,9,11,0,1,15
2,JNJ,2025-11-01,4,9,11,0,1,15
3,JNJ,2025-10-01,3,10,13,0,1,14
4,JNJ,2025-09-01,2,9,14,0,0,13
...,...,...,...,...,...,...,...,...
34905,PCAR,2019-05-01,4,9,14,0,0,17
34906,PCAR,2019-04-01,4,9,14,0,0,17
34907,PCAR,2019-03-01,4,9,14,0,0,17
34908,PCAR,2019-02-01,4,9,14,0,0,17


### Engineering Historical Ratings ###

In [17]:
full_historical_ratings = pd.read_parquet('../../data/raw_data/full_historical_ratings.parquet')
full_historical_ratings = full_historical_ratings.rename(columns={'date': 'Date', 'symbol': 'Ticker'})
full_historical_ratings

ratings = ['S+', 'S', 'S-', 'A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D', 'D-']

rating_ranks = {
    rating: i for i, rating in enumerate(reversed(ratings))
}
full_historical_ratings['rating_rank'] = full_historical_ratings['rating'].map(rating_ranks)
full_historical_ratings['Date'] = pd.to_datetime(full_historical_ratings['Date']).dt.strftime("%Y-%m-%d")
full_historical_ratings.to_parquet('../../data/processed_data/full_historical_ratings.parquet', index=False)   

In [18]:
full_historical_ratings

,Ticker,Date,rating,overallScore,discountedCashFlowScore,returnOnEquityScore,returnOnAssetsScore,debtToEquityScore,priceToEarningsScore,priceToBookScore,rating_rank
0,JNJ,2026-01-22,A-,4,4,5,5,1,3,2,9
1,JNJ,2026-01-21,A-,4,4,5,5,1,3,2,9
2,JNJ,2026-01-20,A-,4,4,5,5,1,3,2,9
3,JNJ,2026-01-16,A-,4,4,5,5,1,3,2,9
4,JNJ,2026-01-15,A-,4,4,5,5,1,3,2,9
...,...,...,...,...,...,...,...,...,...,...,...
2259843,PCAR,2001-01-08,A-,4,5,3,3,4,3,3,9
2259844,PCAR,2001-01-05,A-,4,5,3,3,4,3,3,9
2259845,PCAR,2001-01-04,A-,4,5,3,3,4,3,3,9
2259846,PCAR,2001-01-03,A-,4,5,3,3,4,3,3,9


## Main Chunk to merge back with OHLC ##

In [19]:
daily_ohlc = pd.read_parquet('../../data/raw_data/daily_ohlcv_448_v2.parquet')
daily_ohlc['Date'] = daily_ohlc['Date'].dt.strftime("%Y-%m-%d")

In [20]:
daily_ohlc

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2010-01-04,22.453505,22.625179,22.267525,22.389128,19.856192,3815561.0,A
1,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.418383,493729600.0,AAPL
2,2010-01-04,26.000362,26.177889,25.870815,26.129908,18.321974,10829095.0,ABT
3,2010-01-04,7.978889,8.022222,7.972222,7.994444,7.601905,4813200.0,ACGL
4,2010-01-04,41.520000,42.200001,41.500000,42.070000,31.492170,3650100.0,ACN
...,...,...,...,...,...,...,...,...
1686973,2024-12-31,67.449997,68.059998,67.220001,67.519997,64.848953,2143800.0,XEL
1686974,2024-12-31,106.169998,107.900002,105.779999,107.570000,103.761795,12387800.0,XOM
1686975,2024-12-31,134.089996,134.789993,133.250000,134.160004,131.612030,1217100.0,YUM
1686976,2024-12-31,105.910004,106.500000,104.959999,105.629997,104.596786,683300.0,ZBH


In [21]:
for df in [full_financials, full_financial_ratios, insider_summary, full_williams, 
           historical_analyst_grades,
           full_historical_ratings]:
    
    daily_ohlc = daily_ohlc.merge(
        df,
        on=['Ticker', 'Date'],
        how='left'
    )
    

# Add in the economic data
daily_ohlc = daily_ohlc.merge(
    full_economic_data.rename(columns={'date': 'Date'}),
    on='Date',
    how='left'
)

In [22]:
# Check no duplicates for Date and Ticker columns
assert daily_ohlc.duplicated(subset=['Date', 'Ticker']).sum() == 0, "Duplicates found in Date and Ticker combination!"  

In [23]:
daily_ohlc.to_parquet('../../data/processed_data/full_engineered_dataframe.parquet', index=False)

### Subsetting the Features ###

In [24]:
daily_ohlc.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Ticker',
       'revenue', 'costOfRevenue',
       ...
       'industrialProductionTotalIndex',
       'newPrivatelyOwnedHousingUnitsStartedTotalUnits', 'totalVehicleSales',
       'retailMoneyFunds', 'smoothedUSRecessionProbabilities',
       '3MonthOr90DayRatesAndYieldsCertificatesOfDeposit',
       'commercialBankInterestRateOnCreditCardPlansAllAccounts',
       '30YearFixedRateMortgageAverage', '15YearFixedRateMortgageAverage',
       'tradeBalanceGoodsAndServices'],
      dtype='object', length=237)

In [25]:
### Making the original dupont and financial ratios

def engineer_financial_features(df):
    # Ensure date is datetime and sorted for growth calculations
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['Ticker', 'Date'])
    
    # --- CHANNEL 2: DUPONT RATIOS ---
    # 1. Net Margin: How much profit is generated from each dollar of revenue
    df['dupont_net_margin'] = df['netIncome'] / df['revenue']
    
    # 2. Asset Turnover: How efficiently the company uses assets to generate revenue
    df['dupont_asset_turnover'] = df['revenue'] / df['totalAssets']
    
    # 3. Equity Multiplier: Measures financial leverage
    df['dupont_equity_multiplier'] = df['totalAssets'] / df['totalStockholdersEquity']
    
    # 4. DuPont ROE: The product of the three above
    df['dupont_roe'] = df['dupont_net_margin'] * df['dupont_asset_turnover'] * df['dupont_equity_multiplier']

    # --- CHANNEL 3: CATEGORICAL RATIOS ---
    # 1. Cash Generation & Capital Allocation
    # Measures how much of revenue is converted to Free Cash Flow
    df['cashGenerationCapitalAllocation'] = df['freeCashFlow'] / df['revenue']
    
    # 2. Growth (Year-over-Year Revenue Growth)
    df['growth'] = df.groupby('Ticker')['revenue'].pct_change()
    
    # 3. Leverage (Debt to Equity)
    df['leverage'] = df['totalDebt'] / df['totalStockholdersEquity']
    
    # 4. Profitability (Operating Margin)
    df['profitability'] = df['operatingIncome'] / df['revenue']
    
    # 5. Composite Score (A simplified health score)
    # This combines profitability, growth, and cash flow efficiency
    # We rank them to normalize the scale between 0 and 1
    metrics = ['profitability', 'growth', 'cashGenerationCapitalAllocation']
    df['score'] = df[metrics].rank(pct=True).mean(axis=1)

    # Handle potential infinite values or NaNs from division by zero
    cols_to_fix = ['dupont_net_margin', 'dupont_asset_turnover', 'dupont_equity_multiplier', 
                   'dupont_roe', 'cashGenerationCapitalAllocation', 'growth', 'leverage', 'profitability', 'score']
    df[cols_to_fix] = df[cols_to_fix].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return df

# Apply the function to your dataset
full_data = engineer_financial_features(daily_ohlc)

C:\Users\yinki\AppData\Local\Temp\ipykernel_26800\2883042833.py:27: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['growth'] = df.groupby('Ticker')['revenue'].pct_change()


In [26]:
full_data

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,revenue,costOfRevenue,...,tradeBalanceGoodsAndServices,dupont_net_margin,dupont_asset_turnover,dupont_equity_multiplier,dupont_roe,cashGenerationCapitalAllocation,growth,leverage,profitability,score
0,2010-01-04,22.453505,22.625179,22.267525,22.389128,19.856192,3815561.0,A,NaN,NaN,...,-37744.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
447,2010-01-05,22.324751,22.331903,22.002861,22.145924,19.640507,4186031.0,A,NaN,NaN,...,-37744.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
894,2010-01-06,22.067240,22.174536,22.002861,22.067240,19.570715,3243779.0,A,NaN,NaN,...,-37744.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1341,2010-01-07,22.017166,22.045780,21.816881,22.038628,19.545343,3095172.0,A,NaN,NaN,...,-37744.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1788,2010-01-08,21.917025,22.067240,21.745352,22.031473,19.539000,3733918.0,A,NaN,NaN,...,-37744.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1685189,2024-12-24,390.670013,395.799988,389.059998,395.440002,395.440002,88700.0,ZBRA,NaN,NaN,...,-96948.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.498670
1685636,2024-12-26,392.269989,397.500000,392.109985,396.850006,396.850006,140100.0,ZBRA,NaN,NaN,...,-96948.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.498670
1686083,2024-12-27,393.369995,394.700012,387.010010,389.070007,389.070007,287200.0,ZBRA,NaN,NaN,...,-96948.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.498670
1686530,2024-12-30,385.059998,386.959991,378.149994,383.850006,383.850006,211300.0,ZBRA,NaN,NaN,...,-96948.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.498670


In [28]:
final_features_columns = [
    'Close',
    'dupont_net_margin', 'dupont_asset_turnover', 'dupont_equity_multiplier', 
    'dupont_roe', 'cashGenerationCapitalAllocation', 'growth', 'leverage', 'profitability', 'score',
    'rating_rank',
    'overallScore',
    'analyst_score',
    'totalSecuritiesTransacted', 'totalMarketValue',
]

subset_features = full_data[['Date', 'Ticker'] + final_features_columns]
subset_features.reset_index(drop=True).to_parquet('../../data/processed_data/feature_engineered_data_tickers_448_7_Channels_with_temporal_tape_v2.parquet', index=False)